# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal
import os
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
groq_api_key = os.getenv('GROQ_API_KEY')
groq_url = "https://api.groq.com/openai/v1"
openai = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [4]:
# Some lists!

todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [18]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages, tools=tools, reasoning_effort="high")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [19]:
todos, completed = [], []
loop(messages)

Todo #1: Identify missing quantity: distance between Boston and New York.
Todo #2: Estimate a reasonable distance between Boston and New York.
Todo #3: Compute distance traveled by the Boston train by 3:00 pm.
Todo #4: Compute the remaining distance between the trains at 3:00 pm.
Todo #5: Compute the combined speed of the two trains when traveling towards each other.
Todo #6: Calculate the time after 3:00 pm needed for the trains to meet.
Todo #7: Determine the exact meeting time.
Todo #8: Provide the final answer in a clear format.

The problem does not specify the distance between Boston and New York, which is needed to compute when the trains 
meet.

Todo #1: Identify missing quantity: distance between Boston and New York.
Todo #2: Estimate a reasonable distance between Boston and New York.
Todo #3: Compute distance traveled by the Boston train by 3:00 pm.
Todo #4: Compute the remaining distance between the trains at 3:00 pm.
Todo #5: Compute the combined speed of the two trains when traveling towards each other.
Todo #6: Calculate the time after 3:00 pm needed for the trains to meet.
Todo #7: Determine the exact meeting time.
Todo #8: Provide the final answer in a clear format.

Based on typical geography, the distance between Boston and New York is roughly 200 mi (a convenient round number 
for estimation).

Todo #1: Identify missing quantity: distance between Boston and New York.
Todo #2: Estimate a reasonable distance between Boston and New York.
Todo #3: Compute distance traveled by the Boston train by 3:00 pm.
Todo #4: Compute the remaining distance between the trains at 3:00 pm.
Todo #5: Compute the combined speed of the two trains when traveling towards each other.
Todo #6: Calculate the time after 3:00 pm needed for the trains to meet.
Todo #7: Determine the exact meeting time.
Todo #8: Provide the final answer in a clear format.

...

Time to meet = 140 mi ÷ 140 mph = 1 hour.

Todo #1: Identify missing quantity: distance between Boston and New York.
Todo #2: Estimate a reasonable distance between Boston and New York.
Todo #3: Compute distance traveled by the Boston train by 3:00 pm.
Todo #4: Compute the remaining distance between the trains at 3:00 pm.
Todo #5: Compute the combined speed of the two trains when traveling towards each other.
Todo #6: Calculate the time after 3:00 pm needed for the trains to meet.
Todo #7: Determine the exact meeting time.
Todo #8: Provide the final answer in a clear format.

The trains meet 1 hour after 3:00 pm → at 4:00 pm.

Todo #1: Identify missing quantity: distance between Boston and New York.
Todo #2: Estimate a reasonable distance between Boston and New York.
Todo #3: Compute distance traveled by the Boston train by 3:00 pm.
Todo #4: Compute the remaining distance between the trains at 3:00 pm.
Todo #5: Compute the combined speed of the two trains when traveling towards each other.
Todo #6: Calculate the time after 3:00 pm needed for the trains to meet.
Todo #7: Determine the exact meeting time.
Todo #8: Provide the final answer in a clear format.

**Solution Summary**

1. **Missing quantity** – The distance between Boston and New York is not given.  
2. **Assumed distance** – A reasonable, round‑number estimate for the separation is **200 mi** (the actual distance
is about 215 mi, but 200 mi yields a clean result).  
3. **Boston train travel (2 pm → 3 pm)** – 1 hour × 60 mph = **60 mi**.  
4. **Remaining separation at 3 pm** – 200 mi – 60 mi = **140 mi**.  
5. **Combined closing speed** – 60 mph + 80 mph = **140 mph**.  
6. **Time to meet after 3 pm** – 140 mi ÷ 140 mph = **1 hour**.  
7. **Meeting time** – 3 pm + 1 hour = **4:00 pm**.

---

**Answer:**  
Assuming a ~200 mi distance between the cities, the two trains will meet at **4:00 pm**. (If the true distance 
(≈215 mi) is used, the meeting would occur a few minutes later, around 4:06 pm.)